# Baseline No-Balancing Experiment

Menjalankan 7 model klasifikasi **TANPA balancing** pada **Dataset Kesepakatan** (konsensus 2 annotator).

Data tetap dalam kondisi imbalanced asli (rasio ~27.5:1).

**Output:** 21 runs (7 model x 3 seeds) -> `results/baseline_no_balancing.json`

---

In [ ]:
import os
import json
import numpy as np
import gc
import pandas as pd

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.utils import to_categorical

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import keras_tuner as kt

from src.models import (
    create_mlp_baseline, create_mlp_advance, make_tuner_builder,
    create_naive_bayes, create_svm, create_random_forest,
    create_logistic_regression, get_callbacks, MODEL_CONFIGS
)
from src.balancing import prepare_features

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Konfigurasi

In [ ]:
NUM_RUNS = 3
SEEDS = [42, 123, 456]

print(f"Jumlah runs per model: {NUM_RUNS}")
print(f"Seeds: {SEEDS}")
print(f"Total runs: {len(MODEL_CONFIGS)} models x {NUM_RUNS} seeds = {len(MODEL_CONFIGS) * NUM_RUNS}")

## Load Dataset Kesepakatan

In [ ]:
df_kesepakatan = pd.read_csv('data/processed/preprocessed_dataset_kesepakatan.csv')

print(f"Shape: {df_kesepakatan.shape}")
print(f"\nKolom: {list(df_kesepakatan.columns)}")
print(f"\nDistribusi Kelas:")
print(df_kesepakatan['label'].value_counts())
print(f"\nPersentase:")
print(df_kesepakatan['label'].value_counts(normalize=True) * 100)

## Feature Extraction (TF-IDF)

In [ ]:
X_train_tf, y_one_hot_train, X_test_tf, y_one_hot_test, tfidf = prepare_features(
    df_kesepakatan, 
    "Dataset Kesepakatan", 
    kolom_x='clean_text', 
    kolom_y='label', 
    seed=SEEDS[0]
)

print(f"\nTrain set shape: {X_train_tf.shape}")
print(f"Test set shape: {X_test_tf.shape}")

## Fungsi Helper

In [ ]:
def run_single_baseline(X_train, y_one_hot_train, X_test, y_one_hot_test,
                        model_type, seed, run_idx=0):
    """
    Jalankan 1 kali eksperimen baseline (tanpa balancing).
    """
    callbacks = get_callbacks()

    if model_type == 'mlp_baseline':
        model = create_mlp_baseline(X_train.shape[1])
        model.fit(X_train, y_one_hot_train, epochs=50, batch_size=32,
                  validation_split=0.15, verbose=0)
        y_pred = np.argmax(model.predict(X_test), axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        del model; gc.collect(); K.clear_session()

    elif model_type == 'mlp_advance':
        model = create_mlp_advance(X_train.shape[1])
        model.fit(X_train, y_one_hot_train, epochs=50, batch_size=32,
                  validation_split=0.15, callbacks=callbacks, verbose=0)
        y_pred = np.argmax(model.predict(X_test), axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        del model; gc.collect(); K.clear_session()

    elif model_type == 'mlp_tuner':
        builder = make_tuner_builder(X_train.shape[1])
        tuner = kt.RandomSearch(
            builder, objective='val_accuracy', max_trials=5,
            executions_per_trial=1,
            directory=f'tuner_baseline_run{run_idx}',
            project_name='sentiment',
            overwrite=True
        )
        tuner.search(X_train, y_one_hot_train, epochs=50, batch_size=32,
                     validation_split=0.15, callbacks=callbacks, verbose=0)
        best_model = tuner.get_best_models(1)[0]
        y_pred = np.argmax(best_model.predict(X_test), axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        del best_model, tuner; gc.collect(); K.clear_session()

    elif model_type == 'naive_bayes':
        y_train_1d = np.argmax(y_one_hot_train, axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        model = create_naive_bayes()
        model.fit(X_train, y_train_1d)
        y_pred = model.predict(X_test)

    elif model_type == 'svm':
        y_train_1d = np.argmax(y_one_hot_train, axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        model = create_svm(random_state=seed)
        model.fit(X_train, y_train_1d)
        y_pred = model.predict(X_test)

    elif model_type == 'random_forest':
        y_train_1d = np.argmax(y_one_hot_train, axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        model = create_random_forest(random_state=seed)
        model.fit(X_train, y_train_1d)
        y_pred = model.predict(X_test)

    elif model_type == 'logistic_regression':
        y_train_1d = np.argmax(y_one_hot_train, axis=1)
        y_true = np.argmax(y_one_hot_test, axis=1)
        model = create_logistic_regression(random_state=seed)
        model.fit(X_train, y_train_1d)
        y_pred = model.predict(X_test)

    else:
        raise ValueError(f"model_type tidak dikenal: {model_type}")

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1': f1_score(y_true, y_pred, average='weighted'),
    }


def run_baseline_multi(X_train, y_one_hot_train, X_test, y_one_hot_test,
                       model_type, n_runs=NUM_RUNS, seeds=SEEDS):
    """
    Jalankan 1 model baseline sebanyak n_runs dengan seed berbeda.
    """
    metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

    for run_idx, seed in enumerate(seeds[:n_runs]):
        print(f"  Run {run_idx + 1}/{n_runs} (seed={seed})...", end=' ')

        result = run_single_baseline(
            X_train, y_one_hot_train, X_test, y_one_hot_test,
            model_type, seed, run_idx=run_idx
        )

        for k, v in result.items():
            metrics[k].append(v)

        print(f"Acc={result['accuracy']:.4f} | F1={result['f1']:.4f}")

    summary = {}
    for k, v in metrics.items():
        summary[k] = {
            'mean': np.mean(v),
            'std': np.std(v),
            'runs': v
        }

    print(f"  >> RATA-RATA: Acc={summary['accuracy']['mean']:.4f} "
          f"(+/-{summary['accuracy']['std']:.4f}) | "
          f"F1={summary['f1']['mean']:.4f} "
          f"(+/-{summary['f1']['std']:.4f})")

    return summary


def convert_results(obj):
    """Convert numpy types to native Python types untuk JSON serialization."""
    if isinstance(obj, dict):
        return {k: convert_results(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_results(i) for i in obj]
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj


print("Helper functions defined.")

## Jalankan Eksperimen Baseline

**7 model x 3 seeds = 21 runs**

Estimasi waktu: 1.5 - 3 jam (tergantung CPU/GPU)

In [ ]:
all_results = {'Dataset_Kesepakatan': {'No_Balancing': {}}}

print("=" * 80)
print("MULAI EKSPERIMEN BASELINE NO-BALANCING")
print("=" * 80)

for model_name, model_type in MODEL_CONFIGS.items():
    print(f"\n{'='*80}")
    print(f">> Model: {model_name}")
    print(f"{'='*80}")
    
    all_results['Dataset_Kesepakatan']['No_Balancing'][model_name] = run_baseline_multi(
        X_train_tf, y_one_hot_train, X_test_tf, y_one_hot_test,
        model_type, n_runs=NUM_RUNS, seeds=SEEDS
    )

print("\n\n" + "=" * 80)
print("EKSPERIMEN SELESAI!")
print("=" * 80)

## Simpan Hasil

In [ ]:
os.makedirs('results', exist_ok=True)
output_path = 'results/baseline_no_balancing.json'

with open(output_path, 'w') as f:
    json.dump(convert_results(all_results), f, indent=2)

print(f"Hasil disimpan ke: {output_path}")

## Tampilkan Hasil

In [ ]:
model_names = list(MODEL_CONFIGS.keys())

print("\n" + "=" * 100)
print("BASELINE NO-BALANCING - DATASET KESEPAKATAN")
print("=" * 100)
print(f"{'Model':<22} {'Accuracy':>18} {'Precision':>18} {'Recall':>18} {'F1-Score':>18}")
print("-" * 100)

for model_name in model_names:
    r = all_results['Dataset_Kesepakatan']['No_Balancing'][model_name]
    print(f"{model_name:<22} "
          f"{r['accuracy']['mean']:.4f} +/- {r['accuracy']['std']:.4f}  "
          f"{r['precision']['mean']:.4f} +/- {r['precision']['std']:.4f}  "
          f"{r['recall']['mean']:.4f} +/- {r['recall']['std']:.4f}  "
          f"{r['f1']['mean']:.4f} +/- {r['f1']['std']:.4f}")

## Ringkasan F1-Score (untuk tabel paper)

In [ ]:
print("\n" + "=" * 60)
print("RINGKASAN F1-SCORE WEIGHTED (%)")
print("=" * 60)
print(f"{'Model':<22} {'F1-Score (%)':>20}")
print("-" * 60)

for model_name in model_names:
    r = all_results['Dataset_Kesepakatan']['No_Balancing'][model_name]
    f1_pct = r['f1']['mean'] * 100
    f1_std_pct = r['f1']['std'] * 100
    print(f"{model_name:<22} {f1_pct:>8.2f} +/- {f1_std_pct:.2f}")

print("\nBaris ini akan ditambahkan ke Tabel IV di paper sebagai 'No Bal.'")